# GRF Test Set Trajectory Visualization

Visualize trajectories from GRF test set. Each page shows 5 rows (fields) x 5 columns (time steps).

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

FIELD_NAMES = ['n', 'U', 'vpar', 'psi', 'Ti']
FIELD_LABELS = {
    'n': 'Density (n)',
    'U': 'Vorticity (U)',
    'vpar': 'Parallel velocity (v||)',
    'psi': 'Magnetic flux (psi)',
    'Ti': 'Ion temperature (Ti)'
}

In [ ]:
# ============================================================
# Configuration - MODIFY THESE
# ============================================================
DATA_PATH = '/zhangtao/project2026/OmniFluids/nse2d/data/grf_testset/grf_testset.pt'
N_PAGES = 2          # Number of pages to visualize (5 time steps per page)
TRAJ_IDX = 0         # Which trajectory to visualize (0 to n_samples-1)
SAVE_FIGS = True     # Whether to save figures
OUTPUT_DIR = './vis_grf_traj'  # Output directory for saved figures

In [ ]:
# Load dataset
print(f'Loading: {DATA_PATH}')
data = torch.load(DATA_PATH, map_location='cpu', weights_only=False)

# Print metadata
if 'metadata' in data:
    meta = data['metadata']
    print('\nMetadata:')
    for k, v in meta.items():
        print(f'  {k}: {v}')

# Get data shape
n_field = data['n']
B, T, Nx, Ny = n_field.shape
print(f'\nData shape: B={B} trajectories, T={T} frames, grid={Nx}x{Ny}')

# Get time list
if 't_list' in data:
    t_list = data['t_list'].numpy()
    print(f'Time range: [{t_list[0]:.1f}, {t_list[-1]:.1f}] s')
else:
    t_list = np.arange(T)
    print('No t_list found, using frame indices')

In [ ]:
def visualize_trajectory_page(data, traj_idx, start_frame, n_cols=5, figsize=(20, 16)):
    T_total = data['n'].shape[1]
    end_frame = min(start_frame + n_cols, T_total)
    actual_cols = end_frame - start_frame
    
    fig, axes = plt.subplots(5, actual_cols, figsize=figsize)
    if actual_cols == 1:
        axes = axes.reshape(5, 1)
    
    if 't_list' in data:
        times = data['t_list'].numpy()
    else:
        times = np.arange(T_total)
    
    for col_idx, frame_idx in enumerate(range(start_frame, end_frame)):
        t_val = times[frame_idx]
        
        for row_idx, field_name in enumerate(FIELD_NAMES):
            ax = axes[row_idx, col_idx]
            field = data[field_name][traj_idx, frame_idx].numpy()
            
            vmax = np.abs(field).max()
            if field_name == 'n':
                vmin, vmax = field.min(), field.max()
                cmap = 'viridis'
            else:
                vmin = -vmax
                cmap = 'RdBu_r'
            
            im = ax.imshow(field.T, origin='lower', aspect='auto',
                          cmap=cmap, vmin=vmin, vmax=vmax)
            cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            cbar.ax.tick_params(labelsize=8)
            
            if row_idx == 0:
                ax.set_title(f't = {t_val:.1f}s\n(frame {frame_idx})', fontsize=10)
            if col_idx == 0:
                ax.set_ylabel(FIELD_LABELS[field_name], fontsize=10)
            ax.set_xticks([])
            ax.set_yticks([])
    
    fig.suptitle(f'Trajectory {traj_idx}: Frames {start_frame} - {end_frame-1}', 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    return fig

In [ ]:
# Create output directory if saving
if SAVE_FIGS:
    output_path = Path(OUTPUT_DIR)
    output_path.mkdir(parents=True, exist_ok=True)
    print(f'Figures will be saved to: {output_path}')

# Visualize pages
n_cols_per_page = 5
total_frames = T
max_pages = (total_frames + n_cols_per_page - 1) // n_cols_per_page
actual_pages = min(N_PAGES, max_pages)

print(f'\nVisualizing trajectory {TRAJ_IDX}: {actual_pages} pages ({n_cols_per_page} frames each)')
print(f'Total frames available: {total_frames}')

for page_idx in range(actual_pages):
    start_frame = page_idx * n_cols_per_page
    print(f'\n--- Page {page_idx + 1}/{actual_pages} (frames {start_frame} - {start_frame + n_cols_per_page - 1}) ---')
    
    fig = visualize_trajectory_page(data, TRAJ_IDX, start_frame, n_cols=n_cols_per_page)
    
    if SAVE_FIGS:
        save_path = output_path / f'traj{TRAJ_IDX}_page{page_idx+1}_frames{start_frame}-{start_frame+n_cols_per_page-1}.png'
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f'Saved: {save_path}')
    
    plt.show()

print('\nVisualization complete!')

In [ ]:
# Optional: Visualize field statistics over time
print('Field statistics over time:')

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, field_name in enumerate(FIELD_NAMES):
    ax = axes[i]
    field_traj = data[field_name][TRAJ_IDX]  # (T, Nx, Ny)
    
    means = field_traj.mean(dim=(1, 2)).numpy()
    stds = field_traj.std(dim=(1, 2)).numpy()
    maxs = field_traj.amax(dim=(1, 2)).numpy()
    mins = field_traj.amin(dim=(1, 2)).numpy()
    
    ax.fill_between(t_list, mins, maxs, alpha=0.3, label='min-max')
    ax.fill_between(t_list, means - stds, means + stds, alpha=0.5, label='mean+/-std')
    ax.plot(t_list, means, 'k-', linewidth=1.5, label='mean')
    
    ax.set_xlabel('Time (s)')
    ax.set_ylabel(field_name)
    ax.set_title(FIELD_LABELS[field_name])
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.3)

axes[5].axis('off')
fig.suptitle(f'Field Statistics - Trajectory {TRAJ_IDX}', fontsize=14, fontweight='bold')
plt.tight_layout()

if SAVE_FIGS:
    save_path = output_path / f'traj{TRAJ_IDX}_field_stats.png'
    fig.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f'Saved: {save_path}')

plt.show()